# <font color="steelblue">12. Relación entre variables cualitativas: tablas de contingencia</font>

*Unidad 2. Estadística descriptiva* · Sesión teórica 12 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv"]
PREGUNTAS = ["q_P12_1.json", "q_P12_2.json", "q_P12_3.json", "q_P12_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

¿Los jóvenes se informan más por redes? ¿Quien comparte noticias cae más en bulos? Para responder hay que cruzar dos variables en una **tabla de contingencia**. Saber leerla, y sobre todo saber **qué porcentaje calcular**, separa una buena noticia de un error de base.

**Qué necesitas saber antes**: el cuaderno 5 (frecuencias y porcentajes).

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Construir una tabla de contingencia con `pd.crosstab()`.
2. Calcular porcentajes por filas, por columnas y sobre el total, y elegir el adecuado.
3. Representar la relación con barras apiladas al 100 %.
4. Reconocer cuándo dos variables parecen relacionadas.

## <font color='steelblue'>Contenidos</font>

1. La tabla de contingencia
2. ¿Por filas o por columnas?
3. Gráficos de barras apiladas al 100 %
4. ¿Hay relación?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. La tabla de contingencia</font>

In [ ]:
encuesta = pd.read_csv("encuesta_medios.csv")
encuesta["grupo_edad"] = pd.cut(encuesta["edad"], bins=[17, 29, 44, 64, 100],
                                labels=["18-29", "30-44", "45-64", "65 y más"])

Una **tabla de contingencia** cuenta cuántos casos hay en cada combinación de dos variables. `margins=True` añade los totales.

In [ ]:
pd.crosstab(encuesta["grupo_edad"], encuesta["medio_principal"], margins=True, margins_name="Total")

Con los recuentos brutos cuesta comparar, porque cada grupo de edad tiene un tamaño distinto. Hay que pasar a **porcentajes**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P12_1.json")

## <font color='steelblue'>2. ¿Por filas o por columnas?</font>

La pregunta decide el porcentaje:

- «¿Qué medio usan los jóvenes?» → dentro de cada **grupo de edad** (porcentajes **por fila**, `normalize="index"`).
- «¿Qué edad tiene la audiencia de las redes?» → dentro de cada **medio** (porcentajes **por columna**, `normalize="columns"`).

In [ ]:
por_fila = (pd.crosstab(encuesta["grupo_edad"], encuesta["medio_principal"], normalize="index") * 100).round(1)
por_fila

**Cómo leerlo**: cada fila suma 100. Entre los menores de 30, alrededor del 60 % se informa sobre todo por redes; entre los mayores de 65, apenas un 4 %.

In [ ]:
por_col = (pd.crosstab(encuesta["grupo_edad"], encuesta["medio_principal"], normalize="columns") * 100).round(1)
por_col

**Cómo leerlo**: ahora cada columna suma 100. De todas las personas que se informan por redes, ¿qué parte es joven? Es otra pregunta y otra cifra. Confundirlas es el error de base más común.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P12_2.json")

## <font color='steelblue'>3. Gráficos de barras apiladas al 100 %</font>

Una tabla por filas se representa muy bien con barras apiladas al 100 %: cada barra es un grupo y los colores, su composición.

In [ ]:
por_fila.plot(kind="barh", stacked=True, figsize=(11, 4), colormap="tab10")
plt.xlabel("%")
plt.title("Medio principal según edad (encuesta simulada, n = 1.200)")
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P12_3.json")

## <font color='steelblue'>4. ¿Hay relación?</font>

Si dos variables **no** estuvieran relacionadas, los porcentajes por fila serían parecidos en todas las filas. Si cambian mucho de una fila a otra, hay **asociación**. Ejemplo: ¿quien comparte noticias ha creído más bulos?

In [ ]:
tabla = (pd.crosstab(encuesta["comparte_noticias"], encuesta["ha_creido_bulo"], normalize="index") * 100).round(1)
tabla

**Cómo leerlo**: casi el 39 % de quienes comparten noticias reconoce haber creído un bulo, frente a algo menos del 20 % de quienes no comparten. Hay asociación.

⚠️ **Asociación no es causalidad**: compartir no tiene por qué *causar* creer bulos; puede que ambas cosas dependan de un tercer factor (por ejemplo, informarse por redes). En el cuaderno 30 verás cómo comprobar si una diferencia así puede deberse al azar.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P12_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Haz la tabla de contingencia entre provincia y suscripción de pago, con totales.

<details><summary>Ver solución</summary>

```python
pd.crosstab(encuesta["provincia"], encuesta["suscripcion_pago"], margins=True)
```
</details>

### **Ejercicio 2** *(sección 2)*

¿Qué porcentaje paga por noticias en cada nivel de estudios? Elige bien el tipo de porcentaje.

<details><summary>Ver solución</summary>

```python
(pd.crosstab(encuesta["estudios"], encuesta["suscripcion_pago"], normalize="index") * 100).round(1)
```
</details>

### **Ejercicio 3** *(sección 2)*

Entre quienes han creído un bulo, ¿qué medio principal usan? ¿Es la misma pregunta que «¿qué porcentaje de los que usan redes ha creído un bulo?»?

<details><summary>Ver solución</summary>

```python
print((pd.crosstab(encuesta["ha_creido_bulo"], encuesta["medio_principal"], normalize="index") * 100).round(1))
print((pd.crosstab(encuesta["medio_principal"], encuesta["ha_creido_bulo"], normalize="index") * 100).round(1))
```

No: son dos bases distintas (quienes creyeron un bulo frente a quienes usan cada medio).
</details>

### **Ejercicio 4** *(sección 3)*

Representa con barras apiladas al 100 % la confianza en los medios agrupada en «Baja (0-3)», «Media (4-6)» y «Alta (7-10)» según el medio principal.

<details><summary>Ver solución</summary>

```python
encuesta["conf_grupo"] = pd.cut(encuesta["confianza_medios"], bins=[-1, 3, 6, 10], labels=["Baja", "Media", "Alta"])
(pd.crosstab(encuesta["medio_principal"], encuesta["conf_grupo"], normalize="index") * 100).plot(kind="barh", stacked=True)
plt.show()
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Una **tabla de contingencia** (`pd.crosstab`) cruza dos variables cualitativas.
- La pregunta decide el porcentaje: **por fila** (`normalize="index"`), **por columna** (`"columns"`) o sobre el **total** (`"all"`).
- Confundir la base del porcentaje es el error más común al publicar cruces.
- Si los porcentajes por fila cambian mucho entre filas, hay **asociación**; pero asociación no implica causalidad.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Tabla de contingencia** | Tabla de doble entrada con los recuentos de dos variables cualitativas. |
| **Distribución condicionada** | Distribución de una variable dentro de un grupo de la otra. |
| **Asociación** | Relación estadística entre dos variables. |
| **Marginales** | Totales de filas y columnas. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| Los porcentajes no suman 100 en las filas | Has usado `normalize="columns"` o `"all"` | Usa `normalize="index"` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 13 estudiarás la relación entre variables cuantitativas: correlación y recta de regresión.

**Siguiente cuaderno**: [13. Relación entre variables cuantitativas: correlación y regresión](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P13_correlacion_y_regresion.ipynb)